# n8n Reminders - Descubrimiento dinámico de mensajes

Este notebook demuestra cómo obtener los mensajes de recordatorio de un workflow de n8n
recorriendo dinámicamente el grafo de connections:

```
Switch Ventana → Switch Temperatura N → Mensaje N (n8n-nodes-base.set)
```

## 1. Configuración

In [ ]:
import os
import requests
from typing import Any

N8N_BASE_URL = os.environ.get("N8N_BASE_URL", "https://n8n.ventia-latam.com")
N8N_API_KEY = os.environ["N8N_API_KEY"]  # Set via environment variable
WORKFLOW_ID = "xiypbvTpPjwfyUgu"

headers = {"X-N8N-API-KEY": N8N_API_KEY}
print("Config OK")

## 2. Fetch del workflow completo

In [ ]:
res = requests.get(f"{N8N_BASE_URL}/api/v1/workflows/{WORKFLOW_ID}", headers=headers)
res.raise_for_status()
workflow = res.json()

# Índices para acceso rápido
nodes_by_name: dict[str, dict[str, Any]] = {n["name"]: n for n in workflow["nodes"]}
nodes_by_id: dict[str, dict[str, Any]] = {n["id"]: n for n in workflow["nodes"]}
connections: dict = workflow["connections"]

print(f"Workflow: {workflow['name']}")
print(f"Nodos: {len(workflow['nodes'])}")
print(f"Activo: {workflow['active']}")

## 3. Helpers para recorrer el grafo

In [ ]:
def get_connected_nodes(source_name: str) -> list[list[dict]]:
    """Devuelve los nodos conectados a cada output de un nodo fuente.
    
    Retorna una lista de listas: outputs[i] = [nodos conectados al output i]
    """
    source_conns = connections.get(source_name, {}).get("main", [])
    result = []
    for output_conns in source_conns:
        output_nodes = []
        for conn in output_conns:
            node = nodes_by_name.get(conn["node"])
            if node:
                output_nodes.append(node)
        result.append(output_nodes)
    return result


def get_switch_rules(switch_node: dict) -> list[dict]:
    """Extrae las reglas de un nodo Switch.
    
    Cada regla tiene: output_index, output_key, y el valor contra el que compara.
    """
    rules = switch_node["parameters"].get("rules", {}).get("values", [])
    result = []
    for i, rule in enumerate(rules):
        conditions = rule.get("conditions", {}).get("conditions", [])
        right_value = conditions[0]["rightValue"] if conditions else "unknown"
        result.append({
            "output_index": i,
            "output_key": rule.get("outputKey", f"output_{i}"),
            "value": right_value,
        })
    return result


def get_message_text(node: dict) -> str:
    """Extrae el texto del mensaje de un nodo n8n-nodes-base.set"""
    assignments = node["parameters"].get("assignments", {}).get("assignments", [])
    for a in assignments:
        if a.get("name") == "mensaje":
            text = a.get("value", "")
            # Remover el prefijo "=" que usa n8n para expresiones
            return text.lstrip("=")
    return ""

print("Helpers OK")

## 4. Descubrimiento dinámico: Switch Ventana → Switch Temperatura → Mensajes

In [ ]:
# 1. Encontrar el nodo "Switch Ventana"
switch_ventana = nodes_by_name.get("Switch Ventana")
assert switch_ventana, "No se encontró el nodo 'Switch Ventana' en el workflow"
print(f"Switch Ventana: {switch_ventana['id']}")

# 2. Obtener reglas de ventana (primer_recordatorio, ultimo_recordatorio, etc.)
window_rules = get_switch_rules(switch_ventana)
print(f"Ventanas encontradas: {len(window_rules)}")
for r in window_rules:
    print(f"  - Output {r['output_index']}: {r['value']}")

# 3. Obtener los Switch Temperatura conectados a cada output de Switch Ventana
window_outputs = get_connected_nodes("Switch Ventana")
print(f"\nSwitch Temperatura por ventana:")
for i, nodes in enumerate(window_outputs):
    for n in nodes:
        print(f"  Ventana {i} → {n['name']} ({n['type']})")

## 5. Extraer todos los mensajes (función completa)

In [ ]:
def get_reminder_messages(workflow: dict) -> dict:
    """Recorre el grafo del workflow y extrae todos los mensajes de recordatorio.
    
    Algoritmo:
    1. Buscar "Switch Ventana" 
    2. Por cada output → encontrar el Switch Temperatura correspondiente
    3. Por cada output del Switch Temperatura → encontrar el nodo de mensaje (n8n-nodes-base.set)
    4. Extraer node_id, nombre, temperatura y texto
    """
    nodes_by_name = {n["name"]: n for n in workflow["nodes"]}
    conns = workflow["connections"]
    
    # Encontrar Switch Ventana
    switch_ventana = nodes_by_name.get("Switch Ventana")
    if not switch_ventana:
        return {"windows": [], "workflow_configured": False}
    
    # Reglas de ventana
    window_rules = switch_ventana["parameters"].get("rules", {}).get("values", [])
    
    # Outputs de Switch Ventana → Switch Temperatura N
    ventana_outputs = conns.get("Switch Ventana", {}).get("main", [])
    
    windows = []
    
    for win_idx, output_conns in enumerate(ventana_outputs):
        # Obtener la regla de esta ventana
        win_rule = window_rules[win_idx] if win_idx < len(window_rules) else {}
        win_conditions = win_rule.get("conditions", {}).get("conditions", [])
        win_label = win_conditions[0]["rightValue"] if win_conditions else f"ventana_{win_idx}"
        
        # Encontrar el Switch Temperatura conectado
        switch_temp_name = None
        for conn in output_conns:
            node = nodes_by_name.get(conn["node"])
            if node and node["type"] == "n8n-nodes-base.switch":
                switch_temp_name = conn["node"]
                break
        
        if not switch_temp_name:
            continue
        
        switch_temp = nodes_by_name[switch_temp_name]
        temp_rules = switch_temp["parameters"].get("rules", {}).get("values", [])
        
        # Outputs del Switch Temperatura → nodos de mensaje
        temp_outputs = conns.get(switch_temp_name, {}).get("main", [])
        
        messages = []
        for temp_idx, temp_output_conns in enumerate(temp_outputs):
            # Obtener el valor de temperatura de la regla
            temp_rule = temp_rules[temp_idx] if temp_idx < len(temp_rules) else None
            if not temp_rule:
                continue
                
            temp_conditions = temp_rule.get("conditions", {}).get("conditions", [])
            temperature = temp_conditions[0]["rightValue"] if temp_conditions else f"temp_{temp_idx}"
            
            # Encontrar el nodo de mensaje (tipo n8n-nodes-base.set)
            for conn in temp_output_conns:
                msg_node = nodes_by_name.get(conn["node"])
                if msg_node and msg_node["type"] == "n8n-nodes-base.set":
                    text = ""
                    assignments = msg_node["parameters"].get("assignments", {}).get("assignments", [])
                    for a in assignments:
                        if a.get("name") == "mensaje":
                            text = a.get("value", "").lstrip("=")
                            break
                    
                    messages.append({
                        "node_id": msg_node["id"],
                        "node_name": msg_node["name"],
                        "temperature": temperature,
                        "text": text,
                    })
                    break
        
        windows.append({
            "window": win_idx,
            "window_label": win_label,
            "messages": messages,
        })
    
    return {"windows": windows, "workflow_configured": True}

print("Función get_reminder_messages() definida")

## 6. Ejecutar y ver resultados

In [ ]:
import json

result = get_reminder_messages(workflow)

print(json.dumps(result, indent=2, ensure_ascii=False))

## 7. Vista formateada (como se vería en el dashboard)

In [ ]:
TEMP_COLORS = {
    "frio": "🔵",
    "tibio": "🟡", 
    "caliente": "🔴",
}

for window in result["windows"]:
    print(f"{'='*70}")
    print(f"📋 Ventana {window['window']}: {window['window_label']}")
    print(f"{'='*70}")
    
    for msg in window["messages"]:
        icon = TEMP_COLORS.get(msg["temperature"], "⚪")
        print(f"\n  {icon} {msg['temperature'].upper()}")
        print(f"  Node: {msg['node_name']} ({msg['node_id'][:12]}...)")
        print(f"  Texto: {msg['text'][:100]}{'...' if len(msg['text']) > 100 else ''}")
        print(f"  Chars: {len(msg['text'])}")
    
    print()

total = sum(len(w["messages"]) for w in result["windows"])
print(f"Total: {len(result['windows'])} ventanas, {total} mensajes")

## 8. Simular update de un mensaje (PUT)

In [ ]:
def update_reminder_messages(workflow: dict, updates: list[dict]) -> dict:
    """Actualiza los textos de mensajes en el workflow y hace PUT a n8n.
    
    Args:
        workflow: Workflow completo obtenido del GET
        updates: Lista de {"node_id": "...", "text": "nuevo texto"}
    
    Returns:
        Response de la API de n8n
    """
    nodes_by_id = {n["id"]: n for n in workflow["nodes"]}
    
    # Validar y aplicar cambios
    for update in updates:
        node = nodes_by_id.get(update["node_id"])
        if not node:
            raise ValueError(f"Node ID '{update['node_id']}' no encontrado en el workflow")
        if node["type"] != "n8n-nodes-base.set":
            raise ValueError(f"Node '{node['name']}' no es un nodo de mensaje (tipo: {node['type']})")
        
        # Actualizar el texto (con prefijo "=" que requiere n8n)
        assignments = node["parameters"]["assignments"]["assignments"]
        for a in assignments:
            if a["name"] == "mensaje":
                a["value"] = "=" + update["text"]
                break
    
    # PUT al workflow
    res = requests.put(
        f"{N8N_BASE_URL}/api/v1/workflows/{workflow['id']}",
        headers={**headers, "Content-Type": "application/json"},
        json={
            "name": workflow["name"],
            "nodes": workflow["nodes"],
            "connections": workflow["connections"],
            "settings": workflow["settings"],
        },
    )
    res.raise_for_status()
    return res.json()

print("Función update_reminder_messages() definida")
print("⚠️  La celda siguiente ejecuta el update real — descomentar para probar")

In [ ]:
# ⚠️ DESCOMENTAR PARA EJECUTAR EL UPDATE REAL
# Ejemplo: cambiar el mensaje frío de ventana 1

# updated = update_reminder_messages(workflow, [
#     {
#         "node_id": "e5000001-0000-0000-0000-000000000003",  # Mensaje Frio
#         "text": "Hola! Vi que estabas buscando un producto. ¿Te puedo ayudar?"
#     }
# ])
# print("Workflow actualizado:", updated["updatedAt"])